# Exploration MinIO avec Pandas

Ce notebook se connecte au service **MinIO** via l'API S3 et explore les données du référentiel Anfa.

- MinIO est accessible via `http://minio:9000` (nom DNS Docker Compose)
- Credentials : clé applicative `anfa-app-key` créée en séance 1

In [ ]:
# Installation de boto3 dans le conteneur Jupyter (scipy-notebook n'inclut pas boto3 par défaut)
import subprocess
subprocess.run(["pip", "install", "boto3", "--quiet"], check=True)
print("boto3 installé.")

In [ ]:
import boto3
import pandas as pd
import io

# Connexion à MinIO via l'API S3
# Dans le réseau Docker Compose, MinIO est accessible via http://minio:9000
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="anfa-app-key",
    aws_secret_access_key="anfa-app-secret-2026",
    region_name="us-east-1"
)
print("Connexion à MinIO établie.")

In [ ]:
# Lister les buckets disponibles
response = s3.list_buckets()
print("Buckets disponibles :")
for bucket in response["Buckets"]:
    print(f"  - {bucket['Name']}  (créé le {bucket['CreationDate'].strftime('%Y-%m-%d')})")

In [ ]:
# Lister les objets dans anfa-raw/referentiel/
BUCKET = "anfa-raw"
PREFIX = "referentiel/"

response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
print(f"Objets dans s3://{BUCKET}/{PREFIX}")
for obj in response.get("Contents", []):
    taille_ko = obj["Size"] / 1024
    print(f"  {obj['Key']:45s}  {taille_ko:6.1f} Ko")

In [ ]:
# Charger lignes.csv depuis MinIO dans un DataFrame pandas
obj = s3.get_object(Bucket=BUCKET, Key="referentiel/lignes.csv")
lignes_df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print(f"Dimensions : {lignes_df.shape[0]} lignes × {lignes_df.shape[1]} colonnes")
lignes_df.head()

In [ ]:
# Statistiques descriptives sur les lignes de bus
lignes_df.describe()

In [ ]:
# Visualisation : distance par ligne de bus
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
lignes_df.sort_values("distance_km", ascending=True).plot(
    kind="barh",
    x="nom",
    y="distance_km",
    ax=ax,
    color="steelblue",
    legend=False
)
ax.set_xlabel("Distance (km)")
ax.set_title("Longueur des lignes de bus Anfa")
plt.tight_layout()
plt.savefig("../captures/jupyter-pandas.png", dpi=150)
plt.show()
print("Graphique sauvegardé dans captures/jupyter-pandas.png")

In [ ]:
# Charger et afficher bus.csv
obj = s3.get_object(Bucket=BUCKET, Key="referentiel/bus.csv")
bus_df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print("Répartition des bus par statut :")
print(bus_df["statut"].value_counts().to_string())
print(f"\nCapacité moyenne (bus actifs) : {bus_df[bus_df.statut == 'actif']['capacite'].mean():.1f} places")